# RAG: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Document:** Official Monopoly rulebook (Hasbro, 8-page PDF): `data/rulebook.pdf`

This notebook builds two retrieval pipelines over the **same chunked corpus** and compares them:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranking) |
|---|---|---|
| Stage 1 | Bi-encoder → Top 10 by cosine similarity | same Top 10 |
| Stage 2 | take Top 3 directly | Cross-encoder scores all 10 (query, chunk) pairs → re-sort → Top 3 |
| Generation | LLM → **Answer 1** | same LLM, identical prompt → **Answer 2** |

Then it stress-tests both pipelines with failure-mode queries and profiles latency.

## 1. Setup and imports

In [1]:
# pip install -r requirements.txt
import os, re, time
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv

load_dotenv()  # reads API keys from a local .env file (never committed)

PDF_PATH = "data/rulebook.pdf"

# Model choices
BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker

# Retrieval settings
TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_N_CONTEXT  = 3    # chunks given to the LLM

# LLM provider: "azure", "anthropic" or "openai" (picked automatically from whichever key is set)
LLM_PROVIDER = os.getenv("LLM_PROVIDER") or (
    "azure" if os.getenv("AZURE_OPENAI_API_KEY") else
    "anthropic" if os.getenv("ANTHROPIC_API_KEY") else "openai")
ANTHROPIC_MODEL = "claude-haiku-5-5"
OPENAI_MODEL    = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT") if LLM_PROVIDER == "azure" else "gpt-4o-mini"
print("LLM provider:", LLM_PROVIDER)

pd.set_option("display.max_colwidth", 120)

LLM provider: azure


## 2. Load the PDF
Extract the text page by page with `pypdf` and normalise whitespace. The page number is kept so every chunk can be traced back to its source.

In [2]:
reader = PdfReader(PDF_PATH)
pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()   # collapse newlines / multiple spaces
    pages.append({"page": i + 1, "text": text})

print(f"{len(pages)} pages, {sum(len(p['text']) for p in pages):,} characters")
print(pages[1]["text"][:500])

8 pages, 15,646 characters
Bus: This lets you "get off the bus early." Look at the two white dice. You can move the value of one die, the other die, or the sum of both dice. So if you rolled a 1 and a 5, you can move 1 space, 5 spaces, or 6 spaces: \t's your choice. Mr. Monopoly: First, move the sum of the two white dice and resolve the space you land on (such as drawing a card, buying the property, paying rent, etc.). Then, one of two things will happen depending on whether or not there is still property in the bank. YES


## 3. Chunking
Fixed-size character chunks (~500 chars) with 100-char overlap, so a rule that straddles a boundary is not lost. The cut is moved back to the nearest sentence end where possible. **Both pipelines use exactly this list of chunks.**

In [3]:
CHUNK_SIZE, OVERLAP = 500, 100

def chunk_page(text, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            # try to end on a sentence boundary in the last 40% of the window
            cut = text.rfind(". ", start + int(size * 0.6), end)
            if cut != -1:
                end = cut + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return chunks

chunks = []
for p in pages:
    for c in chunk_page(p["text"]):
        chunks.append({"chunk_id": len(chunks), "page": p["page"], "text": c})

print(f"Total chunks: {len(chunks)}")
pd.DataFrame(chunks).head()

Total chunks: 46


,chunk_id,page,text
0,0,1,"MONOPOLY Property Trading Game from Parker Brothers"" AGES 8+ 2 to 8 Players Contents: Gameboard, 3 dice, tokens, 32 ..."
1,1,1,"ou've never played the classic MONOPOLY game, refer to the Classic Rules beginning on the next page. If you already ..."
2,2,1,yer (two $5005 should work). The game moves fast and you'll need the extra cash to buy and build. 2. Do not use the ...
3,3,1,"l it along with the two white dice on your turn. Then do the following depending on what you rolled. 1, 2, or 3: Add..."
4,4,2,"Bus: This lets you ""get off the bus early."" Look at the two white dice. You can move the value of one die, the other..."


## 4. Index the chunks with a Bi-Encoder (vector store)
Each chunk is embedded **independently** into a 384-d vector. The embeddings are L2-normalised, so the inner product equals cosine similarity. A NumPy matrix is enough as the vector store for a corpus this size: search is a single matrix-vector product.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

chunk_texts = [c["text"] for c in chunks]
t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True)
print(f"Indexed {len(chunk_texts)} chunks -> matrix {chunk_embeddings.shape} in {time.perf_counter()-t0:.2f}s (one-time cost)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Indexed 46 chunks -> matrix (46, 384) in 0.24s (one-time cost)


## 5. Pipeline 1: Baseline retrieval (Top 10 by cosine similarity)

In [5]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Embed the query and return the top-k chunks by cosine similarity, plus the elapsed time."""
    t0 = time.perf_counter()
    q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    sims = chunk_embeddings @ q_emb                 # cosine similarity with every chunk
    top_idx = np.argsort(-sims)[:k]                 # highest first
    elapsed = time.perf_counter() - t0
    results = [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(sims[i])}
               for r, i in enumerate(top_idx)]
    return results, elapsed

## 6. Pipeline 2: Cross-Encoder re-ranking of the same Top 10
The cross-encoder reads the query and the chunk **together** in one forward pass, so attention runs across both texts. It outputs one relevance score per pair, and the candidates are re-sorted by that score.

In [6]:
def rerank(query, candidates):
    """Score every (query, chunk) pair with the cross-encoder and sort descending."""
    t0 = time.perf_counter()
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)
    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, scores)]
    reranked.sort(key=lambda x: x["ce_score"], reverse=True)
    for r, c in enumerate(reranked):
        c["ce_rank"] = r + 1
    elapsed = time.perf_counter() - t0
    return reranked, elapsed

## 7. LLM generation (identical prompt for both pipelines)
The prompt tells the model to answer **only** from the supplied context. This means any difference between Answer 1 and Answer 2 comes from retrieval, not from the prompt.

In [7]:
PROMPT_TEMPLATE = """You are answering questions about the official Monopoly rules.
Use ONLY the context below. If the context does not contain the answer, say "The provided context does not contain this information."
Be concise (2-4 sentences) and cite the chunk ids you used, e.g. [chunk 12].

Context:
{context}

Question: {question}
Answer:"""

def build_context(selected):
    return "\n\n".join(f"[chunk {c['chunk_id']}] {chunks[c['chunk_id']]['text']}" for c in selected)

HAS_KEY = bool(os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY") or os.getenv("OPENAI_API_KEY"))
if not HAS_KEY:
    print("WARNING: no API key found - answers will be placeholders (retrieval/re-ranking still run).")
elif LLM_PROVIDER == "anthropic":
    import anthropic
    _client = anthropic.Anthropic()
elif LLM_PROVIDER == "azure":
    from openai import AzureOpenAI
    _client = AzureOpenAI(api_key=os.getenv("AZURE_OPENAI_API_KEY"),
                          azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
                          api_version=os.getenv("AZURE_OPENAI_API_VERSION"))
else:
    from openai import OpenAI
    _client = OpenAI()

def call_llm(prompt):
    if not HAS_KEY:
        return "[no LLM key set]"
    if LLM_PROVIDER == "anthropic":
        resp = _client.messages.create(model=ANTHROPIC_MODEL, max_tokens=300, temperature=0,
                                       messages=[{"role": "user", "content": prompt}])
        return resp.content[0].text.strip()
    messages = [{"role": "user", "content": prompt}]
    try:
        resp = _client.chat.completions.create(model=OPENAI_MODEL, temperature=0, max_tokens=300, messages=messages)
    except Exception as e:
        # some newer (reasoning) deployments reject temperature / max_tokens -> retry with defaults
        if "unsupported" not in str(e).lower():
            raise
        resp = _client.chat.completions.create(model=OPENAI_MODEL, messages=messages)
    return resp.choices[0].message.content.strip()

def generate_answer(question, selected):
    t0 = time.perf_counter()
    answer = call_llm(PROMPT_TEMPLATE.format(context=build_context(selected), question=question))
    return answer, time.perf_counter() - t0

## 8. Run both pipelines for a query
`run_query` prints a rank table (bi-encoder rank → cross-encoder rank) for all 10 candidates, the two answers, and the timings.

In [8]:
results_log = []   # collects everything for the summary tables

def run_query(query, category="", critical_chunk=None, show=True):
    candidates, t_retrieve = baseline_retrieve(query)
    reranked, t_rerank = rerank(query, candidates)

    top3_base   = candidates[:TOP_N_CONTEXT]
    top3_rerank = reranked[:TOP_N_CONTEXT]

    ans1, t_llm1 = generate_answer(query, top3_base)
    ans2, t_llm2 = generate_answer(query, top3_rerank)

    table = pd.DataFrame([{
        "chunk_id": c["chunk_id"], "page": chunks[c["chunk_id"]]["page"],
        "bi_rank": c["bi_rank"], "ce_rank": c["ce_rank"],
        "shift": c["bi_rank"] - c["ce_rank"],
        "bi_score": round(c["bi_score"], 3), "ce_score": round(c["ce_score"], 2),
        "text": chunks[c["chunk_id"]]["text"][:110] + "...",
    } for c in reranked]).sort_values("bi_rank")

    rec = {"category": category, "query": query, "critical_chunk": critical_chunk,
           "top3_base": [c["chunk_id"] for c in top3_base],
           "top3_rerank": [c["chunk_id"] for c in top3_rerank],
           "answer1": ans1, "answer2": ans2,
           "t_retrieve": t_retrieve, "t_rerank": t_rerank, "t_llm1": t_llm1, "t_llm2": t_llm2,
           "table": table}
    if critical_chunk is not None:
        row = table[table.chunk_id == critical_chunk]
        rec["crit_bi_rank"] = int(row.bi_rank.iloc[0]) if len(row) else None
        rec["crit_ce_rank"] = int(row.ce_rank.iloc[0]) if len(row) else None
    results_log.append(rec)

    if show:
        print(f"QUERY [{category}]: {query}\n")
        print("Baseline top-3 :", rec["top3_base"])
        print("Re-ranked top-3:", rec["top3_rerank"])
        if critical_chunk is not None:
            print(f"Critical chunk {critical_chunk}: rank #{rec['crit_bi_rank']} -> #{rec['crit_ce_rank']}")
        display(table)
        print("ANSWER 1 (baseline):\n", ans1, "\n")
        print("ANSWER 2 (re-ranked):\n", ans2, "\n")
        print(f"Time: retrieve {t_retrieve*1000:.1f} ms | rerank {t_rerank*1000:.1f} ms | "
              f"LLM1 {t_llm1:.2f}s | LLM2 {t_llm2:.2f}s")
    return rec

def show_chunk(cid):
    print(f"[chunk {cid}, page {chunks[cid]['page']}]\n{chunks[cid]['text']}")

## 9. Failure-mode analysis (stress testing)
For each query the **critical chunk** (the chunk that actually contains the answer) was identified by reading the rulebook. We then check where each pipeline ranks it.

### Keyword Trap / Distractor
**Query:** *What happens if two players want to buy the last houses?*

**Critical chunk:** 32. **Why it is hard for the bi-encoder:** The words *players*, *buy* and *money* pull the bi-encoder toward the "No player may borrow from or lend money to another player" chunk. The real answer (houses go to **auction** under BUILDING SHORTAGES) uses different wording.

In [9]:
show_chunk(32)
print()
_ = run_query('What happens if two players want to buy the last houses?', category='Keyword Trap / Distractor', critical_chunk=32)

[chunk 32, page 6]
ing to build must wait for some player to return or sell histher houses to the Bank before building. If there are a limited number of houses and hotels available and two or more players wish to buy more than the Bank has, the houses or hotels must be sold at auction to the highest bidder.



QUERY [Keyword Trap / Distractor]: What happens if two players want to buy the last houses?

Baseline top-3 : [42, 37, 30]
Re-ranked top-3: [32, 37, 5]
Critical chunk 32: rank #5 -> #1


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
4,42,8,1,5,-4,0.600,-0.38,"s) even by selling off buildings and mortgaging property, you must turn over all assets to the Bank. In this c..."
1,37,7,2,2,0,0.579,2.96,"t~es of a color-group are no longer mortgaged, the owner may begin to buy back houses at full price. The playe..."
3,30,6,3,4,-1,0.568,0.08,not build three houses on one property if you have only one house on another property of that group. As you bu...
2,5,2,4,3,1,0.538,1.55,"Then, one of two things will happen depending on whether or not there is still property in the bank. YES, ther..."
0,32,6,5,1,4,0.523,3.77,ing to build must wait for some player to return or sell histher houses to the Bank before building. If there ...
5,18,4,6,6,0,0.516,-2.62,"f you. If you do not wish to buy the property, the Banker sells it at auction to the highest bidder. The buyer..."
7,36,7,7,8,-1,0.509,-8.54,must be sold back to the Bank at half price. The mortgage value is printed on each Title Deed card. No rent ca...
8,34,7,8,9,-1,0.496,-9.48,so located must be sold back to the Bank before the owner can sell any property of that color-group. Houses an...
9,35,7,9,10,-1,0.486,-10.03,"a time (one hotel equals five houses), evenly, in reverse of the manner in which they were erected. MORTGAGES:..."
6,41,8,10,7,3,0.484,-5.54,"islher option, pay the principal or hold the property until some later turn, then lift the mortgage. If helshe..."


ANSWER 1 (baseline):
 The provided context does not contain this information. 

ANSWER 2 (re-ranked):
 If two or more players wish to buy more houses than the Bank has available, the houses must be sold at auction to the highest bidder [chunk 32]. 

Time: retrieve 14.2 ms | rerank 16.3 ms | LLM1 1.80s | LLM2 1.38s


### Keyword Trap / Distractor
**Query:** *What happens if the bank runs out of houses?*

**Critical chunk:** 32. **Why it is hard for the bi-encoder:** "The Bank never goes broke. If the Bank runs out of **money**..." has almost the same phrase. The real rule ("When the Bank has no houses to sell, players... must wait") uses different words.

In [10]:
show_chunk(32)
print()
_ = run_query('What happens if the bank runs out of houses?', category='Keyword Trap / Distractor', critical_chunk=32)

[chunk 32, page 6]
ing to build must wait for some player to return or sell histher houses to the Bank before building. If there are a limited number of houses and hotels available and two or more players wish to buy more than the Bank has, the houses or hotels must be sold at auction to the highest bidder.



QUERY [Keyword Trap / Distractor]: What happens if the bank runs out of houses?

Baseline top-3 : [42, 12, 5]
Re-ranked top-3: [12, 5, 32]
Critical chunk 32: rank #8 -> #3


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
4,42,8,1,5,-4,0.620,-5.21,"s) even by selling off buildings and mortgaging property, you must turn over all assets to the Bank. In this c..."
0,12,3,2,1,1,0.566,5.19,"s all taxes, fines, loans and interest, and the price of all properties which it sells and auctions. The Bank ..."
1,5,2,3,2,1,0.557,-3.09,"Then, one of two things will happen depending on whether or not there is still property in the bank. YES, ther..."
8,35,7,4,9,-5,0.532,-6.59,"a time (one hotel equals five houses), evenly, in reverse of the manner in which they were erected. MORTGAGES:..."
6,34,7,5,7,-2,0.524,-6.19,so located must be sold back to the Bank before the owner can sell any property of that color-group. Houses an...
7,41,8,6,8,-2,0.516,-6.36,"islher option, pay the principal or hold the property until some later turn, then lift the mortgage. If helshe..."
5,36,7,7,6,1,0.509,-6.02,must be sold back to the Bank at half price. The mortgage value is printed on each Title Deed card. No rent ca...
2,32,6,8,3,5,0.498,-3.40,ing to build must wait for some player to return or sell histher houses to the Bank before building. If there ...
9,40,8,9,10,-1,0.497,-8.03,ge for money to the extent of one-half the amount paid for them; this cash is given to the creditor. If you ha...
3,30,6,10,4,6,0.484,-4.77,not build three houses on one property if you have only one house on another property of that group. As you bu...


ANSWER 1 (baseline):
 The provided context does not contain this information. 

ANSWER 2 (re-ranked):
 The provided context does not contain this information. 

Time: retrieve 16.0 ms | rerank 72.3 ms | LLM1 1.10s | LLM2 0.96s


### Negation & Constraints
**Query:** *Is there a limit on the number of hotels on a property?*

**Critical chunk:** 31. **Why it is hard for the bi-encoder:** The constraint "**Only one hotel** may be erected on any one property" sits at the end of the HOTELS chunk. The bi-encoder prefers chunks about house limits.

In [11]:
show_chunk(31)
print()
_ = run_query('Is there a limit on the number of hotels on a property?', category='Negation & Constraints', critical_chunk=31)

[chunk 31, page 6]
te color-group, hdshe may buy a hotel from the Bank and erect it on any property of the color-group. Hdshe returns the four houses from that property to the Bank and pays the price for the hotel as shown on the Ttle Deed card. Only one hotel may be erected on any one property. BUILDING SHORTAGES: When the Bank has no houses to sell, players wishing to build must wait for some player to return or sell histher houses to the Bank before building.



QUERY [Negation & Constraints]: Is there a limit on the number of hotels on a property?

Baseline top-3 : [30, 29, 34]
Re-ranked top-3: [35, 30, 31]
Critical chunk 31: rank #6 -> #3


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
1,30,6,1,2,-1,0.539,-1.98,not build three houses on one property if you have only one house on another property of that group. As you bu...
6,29,6,2,7,-5,0.495,-5.51,ou may buy and erect at any time as many houses as your judgement and financial standing will allow. But you m...
5,34,7,3,6,-3,0.447,-4.74,so located must be sold back to the Bank before the owner can sell any property of that color-group. Houses an...
0,35,7,4,1,3,0.437,-1.97,"a time (one hotel equals five houses), evenly, in reverse of the manner in which they were erected. MORTGAGES:..."
4,20,4,5,5,0,0.391,-4.64,venues) because the owner may then charge double rent for unimproved properties in that color-group. This rule...
2,31,6,6,3,3,0.369,-2.82,"te color-group, hdshe may buy a hotel from the Bank and erect it on any property of the color-group. Hdshe ret..."
3,32,6,7,4,3,0.356,-3.88,ing to build must wait for some player to return or sell histher houses to the Bank before building. If there ...
8,5,2,8,9,-1,0.327,-10.01,"Then, one of two things will happen depending on whether or not there is still property in the bank. YES, ther..."
9,28,6,9,10,-1,0.291,-10.82,erected on one of the unimproved properties of this or any other complete color- group you may own. The price ...
7,11,3,10,8,2,0.272,-8.77,and houses and hotels prior to purchase and use by the players. The Bank pays salaries and bonuses. It sells a...


ANSWER 1 (baseline):
 The provided context does not contain this information. 

ANSWER 2 (re-ranked):
 Yes, there is a limit on the number of hotels on a property; only one hotel may be erected on any one property [chunk 31]. 

Time: retrieve 12.9 ms | rerank 79.9 ms | LLM1 1.00s | LLM2 1.63s


### Exceptions & Corner Cases
**Query:** *What happens to my property if I cannot pay my taxes?*

**Critical chunk:** 41. **Why it is hard for the bi-encoder:** The general rules about taxes and property rank first. The special case ("Should you owe the **Bank**... more than you can pay (because of taxes or penalties)... turn over all assets to the Bank") is in the bankruptcy section.

In [12]:
show_chunk(41)
print()
_ = run_query('What happens to my property if I cannot pay my taxes?', category='Exceptions & Corner Cases', critical_chunk=41)

[chunk 41, page 8]
islher option, pay the principal or hold the property until some later turn, then lift the mortgage. If helshe holds property in this way until a later turn, helshe must pay the interest again upon lifting the mortgage. Should you owe the Bank, instead of another player, more than you can pay (because of taxes or penalties) even by selling off buildings and mortgaging property, you must turn over all assets to the Bank.



QUERY [Exceptions & Corner Cases]: What happens to my property if I cannot pay my taxes?

Baseline top-3 : [5, 36, 17]
Re-ranked top-3: [41, 5, 40]
Critical chunk 41: rank #4 -> #1


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
1,5,2,1,2,-1,0.488,-4.40,"Then, one of two things will happen depending on whether or not there is still property in the bank. YES, ther..."
4,36,7,2,5,-3,0.411,-8.22,must be sold back to the Bank at half price. The mortgage value is printed on each Title Deed card. No rent ca...
5,17,4,3,6,-3,0.399,-9.12,assing GO the first time and another $200 for reaching it the second time by instructions on the card. BUYING ...
0,41,8,4,1,3,0.396,-3.75,"islher option, pay the principal or hold the property until some later turn, then lift the mortgage. If helshe..."
2,40,8,5,3,2,0.379,-5.70,ge for money to the extent of one-half the amount paid for them; this cash is given to the creditor. If you ha...
8,19,4,6,9,-3,0.372,-9.45,"player, the owner collects rent from you in accordance with the list printed on its Title Deed card. If the pr..."
7,42,8,7,8,-1,0.352,-9.35,"s) even by selling off buildings and mortgaging property, you must turn over all assets to the Bank. In this c..."
6,18,4,8,7,1,0.352,-9.15,"f you. If you do not wish to buy the property, the Banker sells it at auction to the highest bidder. The buyer..."
9,27,6,9,10,-1,0.352,-10.56,"""FREE PARKING"": A player landing on this place does not receive any money, property or reward of any kind. Thi..."
3,22,5,10,4,6,0.332,-8.13,"instructions and return the card facedown to the bottom of the deck. The ""Get Out of Jail Free"" card is held u..."


ANSWER 1 (baseline):
 The provided context does not contain this information. 

ANSWER 2 (re-ranked):
 If you owe the Bank more than you can pay, even after selling off buildings and mortgaging property, you must turn over all assets to the Bank [chunk 41]. 

Time: retrieve 11.9 ms | rerank 68.6 ms | LLM1 1.03s | LLM2 1.51s


### Exceptions & Corner Cases (Speed Die)
**Query:** *What happens when I roll three of a kind?*

**Critical chunk:** 6. **Why it is hard for the bi-encoder:** The chunk "1, 2, or 3: Add this number..." overlaps lexically with *three*. The actual special rule ("If you roll a three-of-a-kind... you can move anywhere") is in a "minor details" paragraph.

In [13]:
show_chunk(6)
print()
_ = run_query('What happens when I roll three of a kind?', category='Exceptions & Corner Cases (Speed Die)', critical_chunk=6)

[chunk 6, page 2]
re properties in the bank - Advance to the NOCT property on which you will owe another player money. A few minor details: Only the white dice are used when determining if you rolled doubles. Do not look at the Speed Die. If you roll a three-of-a-kind (all of the dice show the same number), you can move anywhere you want on the board! If you get sent to jail during your move (either by landing on the "Go to Jail" space or by rolling doubles three times in a row) then your turn is over and you do



QUERY [Exceptions & Corner Cases (Speed Die)]: What happens when I roll three of a kind?

Baseline top-3 : [3, 25, 6]
Re-ranked top-3: [6, 7, 26]
Critical chunk 6: rank #3 -> #1


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
4,3,1,1,5,-4,0.401,-4.97,"l it along with the two white dice on your turn. Then do the following depending on what you rolled. 1, 2, or ..."
6,25,5,2,7,-5,0.382,-8.23,"are ""Just Visiting,"" you incur no penalty, and you move ahead in the usual manner on your next turn. You get o..."
0,6,2,3,1,2,0.361,6.30,re properties in the bank - Advance to the NOCT property on which you will owe another player money. A few min...
2,26,5,4,3,1,0.359,-3.82,"you have it; (3) purchasing the ""Get Out of Jail Free"" card from another player and playing it; (4) paying a f..."
5,15,3,5,6,-1,0.359,-7.54,"rties - or obliged to pay rent, pay taxes, draw a Chance or Community Chest card, ""Go to Jail,"" etc. If you th..."
1,7,2,6,2,4,0.331,-3.36,"he ""Go to Jail"" space or by rolling doubles three times in a row) then your turn is over and you do not get to..."
3,4,2,7,4,3,0.294,-4.41,"Bus: This lets you ""get off the bus early."" Look at the two white dice. You can move the value of one die, the..."
8,21,4,8,9,-1,0.290,-11.17,"t collect the rent if he/she fails to ask for it before the second player following throws the dice. ""CHANCE"" ..."
9,13,3,9,10,-1,0.271,-11.22,"ting on any ordinary paper. THE PLAY: Starting with the Banker, each player in turn throws the dice. The playe..."
7,24,5,10,8,2,0.266,-9.26,"Jail""; (2) you draw a card marked ""Go to JailN; or (3) you throw doubles three times in succession. When you a..."


ANSWER 1 (baseline):
 When you roll a three-of-a-kind (all of the dice show the same number), you can move anywhere you want on the board. Your turn continues as normal, but if you get sent to jail during your move, your turn ends [chunk 6]. 

ANSWER 2 (re-ranked):
 When you roll three of a kind (all of the dice show the same number), you can move anywhere you want on the board [chunk 6]. 

Time: retrieve 13.8 ms | rerank 88.1 ms | LLM1 1.85s | LLM2 1.13s


### Control: re-ranker mistake
**Query:** *How much do I pay to unmortgage a property?*

**Critical chunk:** 36. **Why it is hard for the bi-encoder:** Honest counter-example: the cross-encoder promotes the INCOME TAX chunk ("pay 10% of your total worth") because of the shared "pay ... 10%" pattern.

In [14]:
show_chunk(36)
print()
_ = run_query('How much do I pay to unmortgage a property?', category='Control: re-ranker mistake', critical_chunk=36)

[chunk 36, page 7]
must be sold back to the Bank at half price. The mortgage value is printed on each Title Deed card. No rent can be collected on mortgaged properties or utilities, but rent can be collected on unmortgaged properties in the same group. In order to lift the mortgage, the owner must pay the Bank the amount of the mortgage plus 10% interest. When all the propert~es of a color-group are no longer mortgaged, the owner may begin to buy back houses at full price.



QUERY [Control: re-ranker mistake]: How much do I pay to unmortgage a property?

Baseline top-3 : [17, 36, 5]
Re-ranked top-3: [23, 36, 17]
Critical chunk 36: rank #2 -> #2


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
2,17,4,1,3,-2,0.522,-3.21,assing GO the first time and another $200 for reaching it the second time by instructions on the card. BUYING ...
1,36,7,2,2,0,0.508,1.10,must be sold back to the Bank at half price. The mortgage value is printed on each Title Deed card. No rent ca...
9,5,2,3,10,-7,0.415,-10.06,"Then, one of two things will happen depending on whether or not there is still property in the bank. YES, ther..."
0,23,5,4,1,3,0.415,2.93,"may estimate your tax at $900 and pay the Bank, or you may pay 10% of your total worth to the Bank. Your total..."
5,41,8,5,6,-1,0.415,-6.92,"islher option, pay the principal or hold the property until some later turn, then lift the mortgage. If helshe..."
7,18,4,6,8,-2,0.404,-8.74,"f you. If you do not wish to buy the property, the Banker sells it at auction to the highest bidder. The buyer..."
4,28,6,7,5,2,0.404,-5.53,erected on one of the unimproved properties of this or any other complete color- group you may own. The price ...
3,40,8,8,4,4,0.397,-4.96,ge for money to the extent of one-half the amount paid for them; this cash is given to the creditor. If you ha...
6,42,8,9,7,2,0.395,-7.88,"s) even by selling off buildings and mortgaging property, you must turn over all assets to the Bank. In this c..."
8,22,5,10,9,1,0.384,-9.31,"instructions and return the card facedown to the bottom of the deck. The ""Get Out of Jail Free"" card is held u..."


ANSWER 1 (baseline):
 To unmortgage a property, the owner must pay the Bank the amount of the mortgage plus 10% interest [chunk 36]. 

ANSWER 2 (re-ranked):
 To unmortgage a property, the owner must pay the Bank the amount of the mortgage plus 10% interest [chunk 36]. 

Time: retrieve 12.9 ms | rerank 89.4 ms | LLM1 1.27s | LLM2 1.41s


### Control: both agree
**Query:** *Can I collect rent from other players while I am in jail?*

**Critical chunk:** 26. **Why it is hard for the bi-encoder:** Sanity check: the answer chunk is #1 in both pipelines, so re-ranking does not change anything.

In [15]:
show_chunk(26)
print()
_ = run_query('Can I collect rent from other players while I am in jail?', category='Control: both agree', critical_chunk=26)

[chunk 26, page 5]
you have it; (3) purchasing the "Get Out of Jail Free" card from another player and playing it; (4) paying a fine of $50 before you roll the dice on either of your next two turns. If you do not throw doubles by your third turn, you must pay the $50 fine. You then get out of Jail and immediately move forward the number of spaces shown by your throw. Even though you are in Jail, you may buy and sell property, buy and sell houses and hotels and collect rents.



QUERY [Control: both agree]: Can I collect rent from other players while I am in jail?

Baseline top-3 : [26, 22, 24]
Re-ranked top-3: [26, 19, 21]
Critical chunk 26: rank #1 -> #1


,chunk_id,page,bi_rank,ce_rank,shift,bi_score,ce_score,text
0,26,5,1,1,0,0.585,4.18,"you have it; (3) purchasing the ""Get Out of Jail Free"" card from another player and playing it; (4) paying a f..."
6,22,5,2,7,-5,0.483,-4.06,"instructions and return the card facedown to the bottom of the deck. The ""Get Out of Jail Free"" card is held u..."
3,24,5,3,4,-1,0.472,-1.39,"Jail""; (2) you draw a card marked ""Go to JailN; or (3) you throw doubles three times in succession. When you a..."
8,23,5,4,9,-5,0.452,-8.84,"may estimate your tax at $900 and pay the Bank, or you may pay 10% of your total worth to the Bank. Your total..."
5,25,5,5,6,-1,0.446,-4.02,"are ""Just Visiting,"" you incur no penalty, and you move ahead in the usual manner on your next turn. You get o..."
1,19,4,6,2,4,0.446,-0.42,"player, the owner collects rent from you in accordance with the list printed on its Title Deed card. If the pr..."
9,27,6,7,10,-3,0.432,-8.90,"""FREE PARKING"": A player landing on this place does not receive any money, property or reward of any kind. Thi..."
2,21,4,8,3,5,0.416,-0.84,"t collect the rent if he/she fails to ask for it before the second player following throws the dice. ""CHANCE"" ..."
4,20,4,9,5,4,0.409,-3.27,venues) because the owner may then charge double rent for unimproved properties in that color-group. This rule...
7,39,8,10,8,2,0.401,-5.76,BANKRUPTCY.. You are declared bankrupt if you owe more than you can pay either to another player or to the Ban...


ANSWER 1 (baseline):
 Yes, you can collect rent from other players while you are in jail. Even though you are in jail, you may buy and sell property, buy and sell houses and hotels, and collect rents [chunk 26]. 

ANSWER 2 (re-ranked):
 Yes, you can collect rent from other players while you are in jail. Even though you are in jail, you may still buy and sell property, buy and sell houses and hotels, and collect rents [chunk 26]. 

Time: retrieve 14.2 ms | rerank 87.6 ms | LLM1 1.27s | LLM2 5.40s


## 10. Rank-shift summary
Where the critical (answer-bearing) chunk was ranked by the bi-encoder vs. after cross-encoder re-ranking. *In top-3* means the chunk was actually given to the LLM.

In [16]:
summary = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "critical_chunk": r["critical_chunk"],
    "baseline_rank": r.get("crit_bi_rank"), "reranked_rank": r.get("crit_ce_rank"),
    "shift": f"#{r.get('crit_bi_rank')} -> #{r.get('crit_ce_rank')}",
    "in_top3_baseline": (r.get("crit_bi_rank") or 99) <= TOP_N_CONTEXT,
    "in_top3_reranked": (r.get("crit_ce_rank") or 99) <= TOP_N_CONTEXT,
} for r in results_log])
summary

,category,query,critical_chunk,baseline_rank,reranked_rank,shift,in_top3_baseline,in_top3_reranked
0,Keyword Trap / Distractor,What happens if two players want to buy the last houses?,32,5,1,#5 -> #1,False,True
1,Keyword Trap / Distractor,What happens if the bank runs out of houses?,32,8,3,#8 -> #3,False,True
2,Negation & Constraints,Is there a limit on the number of hotels on a property?,31,6,3,#6 -> #3,False,True
3,Exceptions & Corner Cases,What happens to my property if I cannot pay my taxes?,41,4,1,#4 -> #1,False,True
4,Exceptions & Corner Cases (Speed Die),What happens when I roll three of a kind?,6,3,1,#3 -> #1,True,True
5,Control: re-ranker mistake,How much do I pay to unmortgage a property?,36,2,2,#2 -> #2,True,True
6,Control: both agree,Can I collect rent from other players while I am in jail?,26,1,1,#1 -> #1,True,True


### Answer 1 vs. Answer 2 side by side

In [17]:
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame([{"query": r["query"], "Answer 1 (baseline)": r["answer1"], "Answer 2 (re-ranked)": r["answer2"]}
                      for r in results_log]))
pd.set_option("display.max_colwidth", 120)

,query,Answer 1 (baseline),Answer 2 (re-ranked)
0,What happens if two players want to buy the last houses?,The provided context does not contain this information.,"If two or more players wish to buy more houses than the Bank has available, the houses must be sold at auction to the highest bidder [chunk 32]."
1,What happens if the bank runs out of houses?,The provided context does not contain this information.,The provided context does not contain this information.
2,Is there a limit on the number of hotels on a property?,The provided context does not contain this information.,"Yes, there is a limit on the number of hotels on a property; only one hotel may be erected on any one property [chunk 31]."
3,What happens to my property if I cannot pay my taxes?,The provided context does not contain this information.,"If you owe the Bank more than you can pay, even after selling off buildings and mortgaging property, you must turn over all assets to the Bank [chunk 41]."
4,What happens when I roll three of a kind?,"When you roll a three-of-a-kind (all of the dice show the same number), you can move anywhere you want on the board. Your turn continues as normal, but if you get sent to jail during your move, your turn ends [chunk 6].","When you roll three of a kind (all of the dice show the same number), you can move anywhere you want on the board [chunk 6]."
5,How much do I pay to unmortgage a property?,"To unmortgage a property, the owner must pay the Bank the amount of the mortgage plus 10% interest [chunk 36].","To unmortgage a property, the owner must pay the Bank the amount of the mortgage plus 10% interest [chunk 36]."
6,Can I collect rent from other players while I am in jail?,"Yes, you can collect rent from other players while you are in jail. Even though you are in jail, you may buy and sell property, buy and sell houses and hotels, and collect rents [chunk 26].","Yes, you can collect rent from other players while you are in jail. Even though you are in jail, you may still buy and sell property, buy and sell houses and hotels, and collect rents [chunk 26]."


## 11. Latency profiling
Each stage is timed separately and averaged over several runs for all the queries above (after warm-up).
- **Baseline retrieval** = query embedding + cosine search over all chunks.
- **Re-ranking** = cross-encoder over the 10 candidates (extra cost of Pipeline 2).
- **LLM time** is shown separately (network-bound, and the same for both pipelines).

In [18]:
N_RUNS = 5
queries = [r["query"] for r in results_log]

baseline_retrieve(queries[0]); rerank(queries[0], baseline_retrieve(queries[0])[0])   # warm-up

t_ret, t_rr = [], []
for _ in range(N_RUNS):
    for q in queries:
        cands, tr = baseline_retrieve(q)
        _, trr = rerank(q, cands)
        t_ret.append(tr); t_rr.append(trr)

t_llm = [r["t_llm1"] for r in results_log] + [r["t_llm2"] for r in results_log]
ms = lambda xs: np.mean(xs) * 1000

latency = pd.DataFrame([
    {"stage": "Bi-encoder retrieval (top-10)",            "Pipeline 1 (ms)": ms(t_ret), "Pipeline 2 (ms)": ms(t_ret)},
    {"stage": "Cross-encoder re-rank (10 pairs)",         "Pipeline 1 (ms)": 0.0,       "Pipeline 2 (ms)": ms(t_rr)},
    {"stage": "Retrieval total",                          "Pipeline 1 (ms)": ms(t_ret), "Pipeline 2 (ms)": ms(t_ret) + ms(t_rr)},
    {"stage": "LLM generation",                           "Pipeline 1 (ms)": ms(t_llm), "Pipeline 2 (ms)": ms(t_llm)},
    {"stage": "End-to-end",                               "Pipeline 1 (ms)": ms(t_ret) + ms(t_llm), "Pipeline 2 (ms)": ms(t_ret) + ms(t_rr) + ms(t_llm)},
]).round(1)
print(f"Averaged over {len(t_ret)} retrieval runs; re-ranking adds {ms(t_rr)/ms(t_ret):.1f}x the retrieval time")
latency

Averaged over 35 retrieval runs; re-ranking adds 2.8x the retrieval time


,stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Bi-encoder retrieval (top-10),6.8,6.8
1,Cross-encoder re-rank (10 pairs),0.0,19.1
2,Retrieval total,6.8,25.9
3,LLM generation,1625.9,1625.9
4,End-to-end,1632.7,1651.8


### Scaling estimate: cross-encoder over the whole corpus
This times the cross-encoder on **all** chunks for one query and extrapolates to a large collection, to show why it's used only as a second stage.

In [19]:
q = queries[0]
t0 = time.perf_counter()
cross_encoder.predict([(q, c["text"]) for c in chunks])
t_all = time.perf_counter() - t0
per_pair = t_all / len(chunks)

t0 = time.perf_counter(); baseline_retrieve(q); t_bi = time.perf_counter() - t0
print(f"Cross-encoder on all {len(chunks)} chunks: {t_all*1000:.1f} ms  ({per_pair*1000:.2f} ms per pair)")
print(f"Bi-encoder search over all {len(chunks)} chunks: {t_bi*1000:.1f} ms")
for n in [10_000, 1_000_000]:
    print(f"Estimated cross-encoder time for {n:>9,} chunks: {per_pair*n:,.0f} s per query")

Cross-encoder on all 46 chunks: 85.8 ms  (1.86 ms per pair)
Bi-encoder search over all 46 chunks: 4.9 ms
Estimated cross-encoder time for    10,000 chunks: 19 s per query
Estimated cross-encoder time for 1,000,000 chunks: 1,865 s per query


## 12. Observations (raw notes for the report)
- The bi-encoder compresses each chunk into one vector **before** it sees the query. Chunks that share surface words with the query ("runs out of money", "lend money to another player") score high even when they answer a different question.
- The cross-encoder reads query and chunk together, so it can match "buy the last houses" to "limited number of houses... sold at auction" despite the different wording.
- In the 4 queries where the critical chunk was outside the baseline top 3, Answer 1 was "The provided context does not contain this information" (an **omission**: the rule exists, but the LLM never saw it). In 3 of those 4, Answer 2 stated the correct rule and cited the promoted chunk. In the "bank runs out of houses" query the chunk only reached **#3**, behind the "runs out of money" distractor at #1, and the LLM still answered "not found" in this run (it answered correctly in an earlier run, so even at temperature 0 the output is not fully deterministic). Getting a chunk into the top 3 helps, but it does not guarantee the LLM will use it, and rank #1 is much safer than #3.
- In the three-of-a-kind query, Answer 1 added "Your turn continues as normal", which the rulebook never says (a small **hallucination** filled in from loosely related context).
- Re-ranking only helps when the right chunk is already in the top 10. It cannot recover a chunk the bi-encoder missed entirely (recall is set by stage 1).
- The cross-encoder is not perfect: in the unmortgage query it was fooled by the shared "pay 10%" pattern in the Income Tax chunk.
- Latency: re-ranking 10 pairs adds a few tens of milliseconds, which is small next to the LLM call. Running it over every chunk would scale linearly with corpus size, with no index to speed it up.